# Feature Engineering

Prototype the offline feature-engineering pipeline for the Fashion Recommendation System.
This notebook stages sampled Parquet data to S3-compatible storage and builds
**point-in-time** transaction-level features on the **entire** dataset (no train/val/test split).

Each row is featurized using purchase history **strictly before that row's** `t_dat`.

| Stage | Local path | AWS path |
|-------|------------|----------|
| Source datasets | `dataset/dummy`, `dataset/sample_2000_users` | same (uploaded) |
| Staged raw | `s3/dataset/{name}/` | `s3://{bucket}/dataset/{name}/` |
| Features | `s3/dataset/{name}/features/transactions/` | `s3://{bucket}/dataset/{name}/features/transactions/` |

Feature definitions: [`features-eng.md`](../docs/implementation-info/guides/features-eng.md)  
Schema: [`schema-info.md`](../docs/system-design/schema-info.md)  
Requirements: [`v1-requirements.md`](../docs/system-design/v1/v1-requirements.md)

Runs on **local PySpark** (`local[*]`). Configuration lives in `configs/**/*.yaml`;
infrastructure env vars live in `src/fashion_recommendation_system/config.py` (used by
`pipelines/run_feature_pipeline.py`). This notebook loads YAML via `notebooks/config_loader.py`
(does not import from `src/` per project-structure.md).

Core builders live in `notebooks/feature_engineering_core.py` (imported below).

**Kernel:** `Fashion Reco (notebooks)`


## 1. Configuration

Loads merged settings from `configs/data/` and `configs/features/`. To change the active
dataset, edit `configs/data/s3_paths.yaml` → `datasets.active` (e.g. `dummy` for smoke tests).
For AWS/Glue production runs, use `pipelines/run_feature_pipeline.py` (reads `config.py` for infra).

In [1]:
import json
import math
import os
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# --- Path setup: allow running from repo root or notebooks/ directory ---
sys.path.insert(0, str(Path.cwd()))
if not (Path.cwd() / "config_loader.py").exists() and (Path.cwd().parent / "notebooks" / "config_loader.py").exists():
    sys.path.insert(0, str(Path.cwd().parent / "notebooks"))

# Load merged YAML config (paths, dates, feature params) — no src/ imports per project-structure.md
from config_loader import is_glue_runtime, load_feature_engineering_config

CONFIG = load_feature_engineering_config(environment="local_dev")

# Derived flags used throughout the notebook
STORAGE_MODE = CONFIG["storage_mode"]       # "local" → file:///s3/ mirror, "aws" → real S3
STAGE_DATASETS = CONFIG["stage_datasets"]   # e.g. ["dummy", "sample_2000_users"]
IS_GLUE = is_glue_runtime(CONFIG["runtime_mode"])  # True when running on AWS Glue
CONFIG

{'repo_root': 'F:\\git-projects\\fashion-recommendation-system',
 'environment': 'local_dev',
 'storage_mode': 'local',
 'dataset_name': 'sample_2000_users',
 'stage_datasets': ['dummy', 'sample_2000_users'],
 'local_dataset_root': 'dataset',
 'local_s3_root': 's3',
 's3_bucket': 'fashion-reco-dev',
 'aws_region': 'us-east-1',
 'localstack_endpoint': '',
 'train_end': '2020-03-24',
 'cutoff_date': '2020-03-31',
 'test_end': '2020-04-07',
 'feature_cutoff': '2020-03-24',
 'label_window_days': 7,
 'category_col': 'garment_group_name',
 'color_col': 'colour_group_name',
 'decay_half_life_days': 180,
 'item_lookbacks_days': {'short': 7, 'medium': 30, 'long': 180},
 'user_pref_top_n': 3,
 'user_pref_lookback_days': 365,
 'feature_sets': {'item': True,
  'user': True,
  'cross': True,
  'transaction_globals': True},
 'prefixes': {'staged': 'dataset/{dataset_name}',
  'splits': 'dataset/{dataset_name}/splits',
  'features': 'dataset/{dataset_name}/features'},
 'runtime_mode': 'local',
 'cross

## 2. Spark Session

Local PySpark driver (or reuse Glue-provided `spark`). Includes Windows Hadoop shim for
Parquet writes — see [`java-pyspark-local-setup.md`](../docs/implementation-info/guides/java-pyspark-local-setup.md).

In [2]:
# On Glue, the managed SparkSession is already available as `spark`.
if IS_GLUE:
    pass
else:
    import subprocess
    import sys
    from pyspark import SparkContext

    # Pin worker Python to the notebook kernel — avoids version mismatch on Windows.
    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

    # Auto-detect JAVA_HOME on macOS if not set (local dev convenience).
    if not os.environ.get("JAVA_HOME"):
        for java_home_cmd in (
            ["/usr/libexec/java_home", "-v", "17"],
            ["/usr/libexec/java_home", "-v", "1.8"],
            ["/usr/libexec/java_home"],
        ):
            try:
                os.environ["JAVA_HOME"] = subprocess.check_output(
                    java_home_cmd, text=True, stderr=subprocess.DEVNULL
                ).strip()
                break
            except (subprocess.CalledProcessError, FileNotFoundError):
                continue

    def _resolve_repo_root() -> Path:
        for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
            if (candidate / "requirements-notebooks.txt").exists():
                return candidate
        return Path.cwd()

    def _configure_windows_hadoop() -> str | None:
        """Spark local Parquet writes on Windows need winutils (HADOOP_HOME)."""
        if sys.platform != "win32":
            return os.environ.get("HADOOP_HOME")

        def _apply_hadoop_home(hadoop_home: str) -> str:
            os.environ["HADOOP_HOME"] = hadoop_home
            bin_dir = str((Path(hadoop_home) / "bin").resolve())
            if bin_dir not in os.environ.get("PATH", ""):
                os.environ["PATH"] = bin_dir + os.pathsep + os.environ.get("PATH", "")
            return hadoop_home

        hadoop_home = os.environ.get("HADOOP_HOME")
        if hadoop_home and (Path(hadoop_home) / "bin" / "winutils.exe").exists():
            return _apply_hadoop_home(hadoop_home)

        bundled = _resolve_repo_root() / ".hadoop-win"
        if (bundled / "bin" / "winutils.exe").exists():
            return _apply_hadoop_home(str(bundled.resolve()))

        raise RuntimeError(
            "HADOOP_HOME is unset and .hadoop-win/bin/winutils.exe is missing. "
            "See docs/implementation-info/guides/java-pyspark-local-setup.md §5.4."
        )

    HADOOP_HOME = _configure_windows_hadoop()

    def _reset_stale_spark() -> None:
        """Clear cached Spark singletons after kernel restart or JVM crash."""
        SparkSession._instantiatedSession = None
        SparkContext._active_spark_context = None
        SparkContext._gateway = None
        SparkContext._jvm = None

    # Detect dead sessions left over from a prior kernel run.
    try:
        active = SparkSession.getActiveSession()
    except AssertionError:
        active = None
    if active is not None:
        try:
            active.sparkContext._jsc.sc().version()
        except Exception:
            _reset_stale_spark()
    elif SparkSession._instantiatedSession is not None or SparkContext._active_spark_context is not None:
        _reset_stale_spark()

    # Windows: always stop and recreate — avoids file-lock issues on Parquet writes.
    if sys.platform == "win32":
        active = SparkSession.getActiveSession()
        if active is not None:
            active.stop()
            _reset_stale_spark()

    # Local single-machine cluster; tune memory/partitions for laptop-sized datasets.
    builder = (
        SparkSession.builder.appName("feature-engineering")
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .config("spark.sql.shuffle.partitions", "8")
    )
    if HADOOP_HOME:
        # Required on Windows for Parquet writes; also passed through on other OSes.
        builder = (
            builder.config("spark.hadoop.hadoop.home.dir", HADOOP_HOME)
            .config("spark.hadoop.io.native.lib.available", "false")
        )
    spark = builder.getOrCreate()

# Reduce Spark log noise during iterative notebook runs.
spark.sparkContext.setLogLevel("WARN")

## 3. Storage Helpers

Abstract local `s3/` mirror vs real S3 so downstream Spark reads use the same relative keys.

In [3]:
def resolve_repo_root() -> Path:
    """Find repository root by locating requirements-notebooks.txt.

    Returns
    -------
    Path
        Repository root directory.
    """
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "requirements-notebooks.txt").exists():
            return candidate
    return Path.cwd()


def storage_uri(relative_path: str) -> str:
    """Map a relative data-lake key to a local or S3 URI.

    Parameters
    ----------
    relative_path : str
        Path relative to the data-lake root, e.g. ``dataset/sample_2000_users/articles``.

    Returns
    -------
    str
        ``file:///.../s3/...`` in local mode or ``s3://{bucket}/...`` in aws mode.
    """
    relative_path = relative_path.strip("/").replace("\\", "/")
    if STORAGE_MODE == "aws":
        # Production: read/write against the real S3 bucket.
        return f"s3://{CONFIG['s3_bucket']}/{relative_path}"

    # Local dev: mirror S3 layout under repo/s3/ so Spark uses file:// URIs.
    local_root = (resolve_repo_root() / CONFIG["local_s3_root"]).resolve()
    return str((local_root / relative_path).resolve())


def local_source_dataset_path(dataset_name: str) -> Path:
    """Resolve on-disk Parquet source under ``dataset/``.

    Parameters
    ----------
    dataset_name : str
        Subfolder name, e.g. ``dummy`` or ``sample_2000_users``.

    Returns
    -------
    Path
        Absolute path to ``dataset/{dataset_name}``.
    """
    return (resolve_repo_root() / CONFIG["local_dataset_root"] / dataset_name).resolve()


def copy_tree_local(src: Path, dest: Path) -> None:
    """Recursively copy a directory tree (idempotent overwrite).

    Parameters
    ----------
    src : Path
        Source directory.
    dest : Path
        Destination directory (created if missing).
    """
    if not src.exists():
        raise FileNotFoundError(f"Source dataset not found: {src}")
    # Idempotent: wipe destination so re-runs don't leave stale Parquet parts.
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(src, dest)


def upload_tree_to_s3(local_src: Path, s3_prefix: str) -> int:
    """Upload a local directory tree to S3 using boto3.

    Parameters
    ----------
    local_src : Path
        Local folder containing Parquet partitions.
    s3_prefix : str
        Key prefix inside the bucket, e.g. ``dataset/dummy``.

    Returns
    -------
    int
        Number of files uploaded.
    """
    import boto3

    if not local_src.exists():
        raise FileNotFoundError(f"Source dataset not found: {local_src}")

    client_kwargs = {"region_name": CONFIG["aws_region"]}
    if CONFIG["localstack_endpoint"]:
        # Point boto3 at LocalStack when testing S3 locally.
        client_kwargs["endpoint_url"] = CONFIG["localstack_endpoint"]

    s3 = boto3.client("s3", **client_kwargs)
    uploaded = 0
    for path in local_src.rglob("*"):
        if path.is_file():
            key = f"{s3_prefix.strip('/')}/{path.relative_to(local_src).as_posix()}"
            s3.upload_file(str(path), CONFIG["s3_bucket"], key)
            uploaded += 1
    return uploaded


def stage_dataset(dataset_name: str) -> dict:
    """Stage one dataset from ``dataset/`` to S3-compatible storage.

    Local mode copies to ``s3/dataset/{name}/``. AWS mode uploads to
    ``s3://{bucket}/dataset/{name}/``.

    Parameters
    ----------
    dataset_name : str
        Dataset folder name under ``dataset/``.

    Returns
    -------
    dict
        Summary with ``dataset``, ``mode``, ``source``, ``destination``, ``files``.
    """
    src = local_source_dataset_path(dataset_name)
    relative_dest = f"dataset/{dataset_name}"

    if STORAGE_MODE == "aws":
        files = upload_tree_to_s3(src, relative_dest)
        destination = storage_uri(relative_dest)
    else:
        # Local mode: copy dataset/ → s3/dataset/ to mimic the data-lake layout.
        dest = Path(storage_uri(relative_dest))
        copy_tree_local(src, dest)
        files = sum(1 for p in dest.rglob("*") if p.is_file())
        destination = str(dest)

    return {
        "dataset": dataset_name,
        "mode": STORAGE_MODE,
        "source": str(src),
        "destination": destination,
        "files": files,
    }

### 3.1 Stage datasets to S3

Copy `dataset/dummy` and `dataset/sample_2000_users` into the S3-compatible layout.
Re-running overwrites the destination (idempotent).

In [4]:
# Stage every dataset listed in config (typically dummy + sample_2000_users).
stage_results = [stage_dataset(name) for name in STAGE_DATASETS]
stage_results

[{'dataset': 'dummy',
  'mode': 'local',
  'source': 'F:\\git-projects\\fashion-recommendation-system\\dataset\\dummy',
  'destination': 'F:\\git-projects\\fashion-recommendation-system\\s3\\dataset\\dummy',
  'files': 38},
 {'dataset': 'sample_2000_users',
  'mode': 'local',
  'source': 'F:\\git-projects\\fashion-recommendation-system\\dataset\\sample_2000_users',
  'destination': 'F:\\git-projects\\fashion-recommendation-system\\s3\\dataset\\sample_2000_users',
  'files': 146}]

## 4. Load Staged Data

Read Parquet tables from the staged location for the active dataset (`CONFIG['dataset_name']`).

In [5]:
def read_staged_table(spark: SparkSession, dataset_name: str, table: str) -> DataFrame:
    """Read a staged Parquet table (articles, customers, or transactions).

    Parameters
    ----------
    spark : SparkSession
        Active Spark session.
    dataset_name : str
        Dataset folder name under ``dataset/``.
    table : str
        Table subfolder: ``articles``, ``customers``, or ``transactions``.

    Returns
    -------
    DataFrame
        Parquet DataFrame with schema from the sampling notebook.
    """
    path = storage_uri(f"dataset/{dataset_name}/{table}")
    # Spark resolves file:// or s3:// via storage_uri — same code path for local and AWS.
    return spark.read.parquet(path)


# Active dataset comes from configs/data/s3_paths.yaml → datasets.active
DATASET = CONFIG["dataset_name"]

# Read the three raw Parquet tables from the staged data-lake path.
articles_df = read_staged_table(spark, DATASET, "articles")
customers_df = read_staged_table(spark, DATASET, "customers")
transactions_df = read_staged_table(spark, DATASET, "transactions")

# Row counts — sanity check before feature work.
print(f"articles: {articles_df.count():,}")
print(f"customers: {customers_df.count():,}")
print(f"transactions: {transactions_df.count():,}")

articles: 22,186
customers: 2,000
transactions: 48,265


### 4.1 Parquet write helpers

Shared helpers for persisting enriched transaction features.

In [6]:
def write_parquet_dataset(
    df: DataFrame,
    relative_path: str,
    partition_cols: list[str] | None = None,
) -> str:
    """Write a DataFrame as Parquet to the configured storage backend.

    Parameters
    ----------
    df : DataFrame
        Data to persist.
    relative_path : str
        Destination key relative to the data-lake root.
    partition_cols : list[str] | None
        Optional Hive partition columns.

    Returns
    -------
    str
        Resolved destination URI/path.
    """
    dest = storage_uri(relative_path)
    # Local file writes: remove existing folder so overwrite is clean.
    if STORAGE_MODE == "local":
        dest_path = Path(dest)
        if dest_path.exists():
            shutil.rmtree(dest_path)

    writer = df.write.mode("overwrite")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.parquet(dest)
    return dest


def add_transaction_partitions(df: DataFrame) -> DataFrame:
    """Add year/month Hive partition columns for transactions.

    Parameters
    ----------
    df : DataFrame
        Transactions with ``t_dat``.

    Returns
    -------
    DataFrame
        Input plus ``year`` and ``month`` string columns.
    """
    # Hive-style partitions speed up time-range reads in downstream jobs.
    return df.withColumn("year", F.date_format("t_dat", "yyyy")).withColumn(
        "month", F.date_format("t_dat", "MM")
    )

## 5. Point-in-Time Feature Engineering

1. Merge item attributes (`item_category`, `item_color`) and user keys onto transactions.
2. For each transaction row, compute item, user, cross, and time features using only
   prior history relative to that row's `t_dat` (same-day tie-break via `txn_id`).
3. Write one enriched transactions table under `features/transactions/`.

Key conventions:
- `item_recent_to_last_180d_ratio` = `item_pop_30d / (item_category_pop_180d + 1)`
- `item_seasonality_strength` = 7d item pop vs the same 7-day window one year earlier
- `user_color_pref_1y_rank1/2` only (no rank3)
- `txn_month_sin` / `txn_month_cos` from the row's purchase month


In [7]:
from feature_engineering_core import build_enriched_transactions

enriched_txn_df = build_enriched_transactions(
    transactions_df,
    articles_df,
    customers_df,
    CONFIG["category_col"],
    CONFIG["color_col"],
    CONFIG["decay_half_life_days"],
    color_pref_top_n=2,
    category_pref_top_n=CONFIG["user_pref_top_n"],
)

print(f"enriched transactions: {enriched_txn_df.count():,}")
enriched_txn_df.select(
    "customer_id",
    "article_id",
    "t_dat",
    "item_pop_7d",
    "item_recent_to_last_180d_ratio",
    "item_seasonality_strength",
    "user_color_pref_1y_rank1",
    "user_color_pref_1y_rank2",
    "txn_month_sin",
    "txn_month_cos",
).show(5, truncate=False)


enriched transactions: 48,265
+----------------------------------------------------------------+----------+----------+-----------+------------------------------+-------------------------+------------------------+------------------------+-----------------------+-------------+
|customer_id                                                     |article_id|t_dat     |item_pop_7d|item_recent_to_last_180d_ratio|item_seasonality_strength|user_color_pref_1y_rank1|user_color_pref_1y_rank2|txn_month_sin          |txn_month_cos|
+----------------------------------------------------------------+----------+----------+-----------+------------------------------+-------------------------+------------------------+------------------------+-----------------------+-------------+
|01087f97e0d501b68c8723c60dece1f28a6efbc5b6cc0df9abe8a566a8c0c036|0573085033|2019-12-01|0          |0.0                           |0.0                      |Black                   |Blue                    |-2.4492935982947064E-16|1

## 6. Persist Feature Outputs

Write the enriched transaction table under `dataset/{name}/features/transactions/`.


In [8]:
features_base = f"dataset/{DATASET}/features"
enriched_partitioned = add_transaction_partitions(enriched_txn_df)
feature_outputs = {
    "transactions": write_parquet_dataset(
        enriched_partitioned,
        f"{features_base}/transactions",
        partition_cols=["year", "month"],
    ),
}
feature_outputs


{'transactions': 'F:\\git-projects\\fashion-recommendation-system\\s3\\dataset\\sample_2000_users\\features\\transactions'}

## 7. Run Summary


In [9]:
summary = {
    "run_at_utc": datetime.now(timezone.utc).isoformat(),
    "storage_mode": CONFIG["storage_mode"],
    "dataset": DATASET,
    "transaction_rows": enriched_txn_df.count(),
    "feature_columns": enriched_txn_df.columns,
    "staged_datasets": stage_results,
    "feature_outputs": feature_outputs,
}
print(json.dumps(summary, indent=2))


{
  "run_at_utc": "2026-06-10T20:23:52.523538+00:00",
  "storage_mode": "local",
  "dataset": "sample_2000_users",
  "transaction_rows": 48265,
  "feature_columns": [
    "customer_id",
    "article_id",
    "t_dat",
    "price",
    "item_category",
    "item_color",
    "item_pop_7d",
    "item_pop_30d",
    "item_pop_180d",
    "item_category_pop_30d",
    "item_category_pop_180d",
    "item_pop_same_7d_last_year",
    "first_sold_date",
    "days_since_first_sold",
    "item_recent_to_last_180d_ratio",
    "item_category_recent_to_lifetime_ratio",
    "item_seasonality_strength",
    "user_category_pref_1y_rank1",
    "user_category_pref_1y_rank2",
    "user_category_pref_1y_rank3",
    "user_color_pref_1y_rank1",
    "user_color_pref_1y_rank2",
    "user_days_since_last_purchase",
    "user_purchase_count_30d",
    "user_purchase_count_180d",
    "user_decayed_price_avg",
    "user_decayed_price_std",
    "user_item_repurchase",
    "user_item_decayed_repurchase",
    "user_item_d